In [1]:
import torch

# 历史输入：总用户数、预约人数、播出时间、推送方式
input = torch.tensor([
    [30000, 1800, 14, 1],
    [40000, 2000, 16, 1],
    [50000, 3200, 20, 2],
    [60000, 2500, 12, 0],
    [70000, 3000, 15, 0],
    [80000, 4200, 18, 1],
    [100000, 6000, 20, 2],
    [120000, 7500, 21, 2],
    [150000, 9000, 19, 2],
    [200000, 15000, 22, 2]
], dtype=torch.float32)

# 历史输出：在线人数峰值
output = torch.tensor([4100, 4300, 9000, 2800, 3600, 9100, 15800, 19200, 22500, 38000], dtype=torch.float32).reshape(-1, 1)

def model(input, weights, bias):
    return torch.matmul(input, weights) + bias

weights = torch.normal(0, 0.01, size=(4,1), requires_grad=True)
bias = torch.zeros(1, requires_grad=True)

def loss(pred, target):
    return ((pred - target) ** 2).mean() # 自动求导只能针对标量，咱们用个 `mean()` 求均值，把 loss 返回的矩阵转成标量

def optimizer(weights, bias, learning_rate=1e-9):
    with torch.no_grad():
        weights -= weights.grad * learning_rate # 矩阵减去常数时，Torch 会自动帮我们把矩阵里的每个参数都减去那个常数，它管这种行为叫「广播」
        weights.grad.zero_()
        bias -= bias.grad * learning_rate
        bias.grad.zero_()

for epoch in range(100):
    pred = model(input, weights, bias)
    l = loss(pred, output)
    l.backward()
    optimizer(weights, bias)

    print(f'Epoch {epoch}: Loss = {l.item():.2f}')

Epoch 0: Loss = 249500256.00
Epoch 1: Loss = 97667743744.00
Epoch 2: Loss = 40828781199360.00
Epoch 3: Loss = 17070723327590400.00
Epoch 4: Loss = 7137358785428848640.00
Epoch 5: Loss = 2984167303788890161152.00
Epoch 6: Loss = 1247696263971548224815104.00
Epoch 7: Loss = 521668388381284006834995200.00
Epoch 8: Loss = 218112318533618933056594247680.00
Epoch 9: Loss = 91193897505865684990050932097024.00
Epoch 10: Loss = 38128652245192855297315018029137920.00
Epoch 11: Loss = 15941788871482579631729920992140591104.00
Epoch 12: Loss = inf
Epoch 13: Loss = inf
Epoch 14: Loss = inf
Epoch 15: Loss = inf
Epoch 16: Loss = inf
Epoch 17: Loss = inf
Epoch 18: Loss = inf
Epoch 19: Loss = inf
Epoch 20: Loss = inf
Epoch 21: Loss = inf
Epoch 22: Loss = inf
Epoch 23: Loss = inf
Epoch 24: Loss = inf
Epoch 25: Loss = nan
Epoch 26: Loss = nan
Epoch 27: Loss = nan
Epoch 28: Loss = nan
Epoch 29: Loss = nan
Epoch 30: Loss = nan
Epoch 31: Loss = nan
Epoch 32: Loss = nan
Epoch 33: Loss = nan
Epoch 34: Loss = 

In [2]:
mean = input.mean(dim=0) # 先求均值，dim=0 表示保留第一个维度，即求每个列的均值
std = input.std(dim=0) # 再求出标准差，dim=0 表示保留第一个维度，即求每个列的标准差
std = torch.where(std == 0, torch.ones_like(std), std) # where 是 torch 的三元运算符，这句意思是如果等于 0 则等于 1。因为后面要算“几个标准差”，所以要除以标准差，为防止出现除0错误，防一手

input_zscore = (input - mean) / std # 我们新的数据集

print(mean)
print(std)
print(input_zscore)

# 以上三行 print 会输出：
# tensor([9.0000e+04, 5.4200e+03, 1.7700e+01, 1.3000e+00])
# tensor([5.3541e+04, 4.1480e+03, 3.3015e+00, 8.2327e-01])
# tensor([[-1.1206, -0.8727, -1.1207, -0.3644],
#         [-0.9339, -0.8245, -0.5149, -0.3644],
#         [-0.7471, -0.5352,  0.6966,  0.8503],
#         [-0.5603, -0.7039, -1.7265, -1.5791],
#         [-0.3735, -0.5834, -0.8178, -1.5791],
#         [-0.1868, -0.2941,  0.0909, -0.3644],
#         [ 0.1868,  0.1398,  0.6966,  0.8503],
#         [ 0.5603,  0.5014,  0.9995,  0.8503],
#         [ 1.1206,  0.8631,  0.3938,  0.8503],
#         [ 2.0545,  2.3095,  1.3024,  0.8503]])

weights = torch.normal(0, 0.01, size=(4,1), requires_grad=True)
bias = torch.zeros(1, requires_grad=True)

for epoch in range(20000):
    pred = model(input_zscore, weights, bias)
    l = loss(pred, output)
    l.backward()
    optimizer(weights, bias, learning_rate=1e-2)

    if epoch % 10 == 0:
        print(f'Epoch {epoch}: Loss = {l.item():.2f}')

tensor([9.0000e+04, 5.4200e+03, 1.7700e+01, 1.3000e+00])
tensor([5.3541e+04, 4.1480e+03, 3.3015e+00, 8.2327e-01])
tensor([[-1.1206, -0.8727, -1.1207, -0.3644],
        [-0.9339, -0.8245, -0.5149, -0.3644],
        [-0.7471, -0.5352,  0.6966,  0.8503],
        [-0.5603, -0.7039, -1.7265, -1.5791],
        [-0.3735, -0.5834, -0.8178, -1.5791],
        [-0.1868, -0.2941,  0.0909, -0.3644],
        [ 0.1868,  0.1398,  0.6966,  0.8503],
        [ 0.5603,  0.5014,  0.9995,  0.8503],
        [ 1.1206,  0.8631,  0.3938,  0.8503],
        [ 2.0545,  2.3095,  1.3024,  0.8503]])
Epoch 0: Loss = 278843776.00
Epoch 10: Loss = 148628528.00
Epoch 20: Loss = 88782840.00
Epoch 30: Loss = 56874188.00
Epoch 40: Loss = 37919656.00
Epoch 50: Loss = 25916972.00
Epoch 60: Loss = 18056420.00
Epoch 70: Loss = 12819421.00
Epoch 80: Loss = 9297670.00
Epoch 90: Loss = 6915138.50
Epoch 100: Loss = 5295346.50
Epoch 110: Loss = 4188568.50
Epoch 120: Loss = 3427935.50
Epoch 130: Loss = 2901489.75
Epoch 140: Loss = 25

In [3]:
def inference(raw_input, weights, bias, mean, std):
    normalized_input = (raw_input - mean) / std
    with torch.no_grad():
        pred = model(normalized_input, weights, bias)
    return pred.squeeze().item()

true_values = output.flatten().tolist()
for i in range(len(input)):
    raw_x = input[i]
    true_y = true_values[i]
    pred_y = inference(raw_x, weights, bias, mean, std)
    print(f"样本 {i+1:2d} | 真实值: {true_y:6.0f} | 预测值: {pred_y:8.1f} | 误差: {pred_y - true_y:8.1f}")

new_sample = torch.tensor([180000, 12000, 20, 2], dtype=torch.float32).unsqueeze(0)  # 总用户18万，预约1.2万，20点开播，短信推送
pred_new = inference(new_sample, weights, bias, mean, std)
print(f"输入: 总用户={int(new_sample[0,0])}, 预约={int(new_sample[0,1])}, 时间={int(new_sample[0,2])}点, 推送={int(new_sample[0,3])}")
print(f"预测最高在线人数: {pred_new:.0f}")

样本  1 | 真实值:   4100 | 预测值:   4058.9 | 误差:    -41.1
样本  2 | 真实值:   4300 | 预测值:   4136.1 | 误差:   -163.9
样本  3 | 真实值:   9000 | 预测值:   9279.3 | 误差:    279.3
样本  4 | 真实值:   2800 | 预测值:   2854.6 | 误差:     54.6
样本  5 | 真实值:   3600 | 预测值:   3725.9 | 误差:    125.9
样本  6 | 真实值:   9100 | 预测值:   8943.4 | 误差:   -156.6
样本  7 | 真实值:  15800 | 预测值:  15618.8 | 误差:   -181.2
样本  8 | 真实值:  19200 | 预测值:  19180.2 | 误差:    -19.8
样本  9 | 真实值:  22500 | 预测值:  22611.5 | 误差:    111.5
样本 10 | 真实值:  38000 | 预测值:  37991.1 | 误差:     -8.9
输入: 总用户=180000, 预约=12000, 时间=20点, 推送=2
预测最高在线人数: 30162
